# FightWatch — sửa cận cảnh và hành vi dễ nhầm
Một cấu hình RGB từ checkpoint đã phục hồi, không train lại hai kiến trúc. Bổ sung 251 cửa sổ âm public thuộc 42 nhóm biểu diễn; không lấy development/calibration/test hay video iPhone làm train. Chọn checkpoint theo sự kiện development, không theo loss đơn thuần. Tối đa 45 phút GPU toàn phiên, dừng sớm nếu ba epoch không cải thiện; không tự cài model vào ứng dụng.

Chỉ chạy sau khi duyệt ngân sách GPU mới. Chạy tất cả cell; artifact và checkpoint ở Drive, cache được giữ lại khi dừng. `--resume` chỉ chấp nhận đúng dữ liệu/code/parent checkpoint; thay code thì dùng thư mục output mới. Việc đã xem các split public không biến chúng thành nghiệm thu camera độc lập.

In [ ]:
import time, threading
from pathlib import Path
import torch
from google.colab import drive, runtime
assert torch.cuda.is_available(), 'Chọn GPU sau khi ngân sách được duyệt.'
if 'CLASSROOM_GPU_DEADLINE' not in globals():
    CLASSROOM_GPU_DEADLINE = time.monotonic() + 45*60
    CLASSROOM_WATCHDOG = threading.Timer(45*60, runtime.unassign)
    CLASSROOM_WATCHDOG.daemon = True
    CLASSROOM_WATCHDOG.start()
try:
    drive.mount('/content/drive', timeout_ms=180000)
    ROOT = Path('/content/drive/MyDrive/SENTINEL')
    REPO = ROOT/'repo/anomaly-behavior-detector-system'
    SOURCE = ROOT/'artifacts/fight_fusion_v1'
    REVIEW = ROOT/'artifacts/fight_temporal_review_v1'
    PARENT = ROOT/'artifacts/fight_refinement_v2'
    OUTPUT = ROOT/'artifacts/fight_classroom_refinement_v1'
    CACHE = Path('/content/classroom-window-cache')
    DATA = Path('/content/fightwatch-reviewed-public-data')
    LOCAL = Path('/content/fightwatch-classroom-code')
    OUTPUT.mkdir(parents=True, exist_ok=True)
    def remaining(reserve=0):
        seconds = CLASSROOM_GPU_DEADLINE - time.monotonic() - reserve
        if seconds <= 0:
            raise TimeoutError('Hết ngân sách GPU toàn phiên; artifact trên Drive được giữ nguyên.')
        return seconds
    print('GPU:', torch.cuda.get_device_name(0), 'Không reset watchdog khi chạy lại cell.')
except BaseException:
    runtime.unassign()
    raise


In [ ]:
import os, sys, json, shutil, subprocess, zipfile
try:
    subprocess.run(['git','-C',str(REPO),'restore','--','colab'], check=True, timeout=min(60,remaining()))
    dirty = subprocess.check_output(['git','-C',str(REPO),'status','--porcelain','--untracked-files=no'], text=True, timeout=min(60,remaining()))
    assert not dirty.strip(), 'Có thay đổi code tracked trên Drive: '+dirty
    subprocess.run(['git','-C',str(REPO),'pull','--ff-only','origin','main'], check=True, timeout=min(90,remaining()))
    REVISION = subprocess.check_output(['git','-C',str(REPO),'rev-parse','HEAD'],text=True,timeout=30).strip()
    snapshot = Path('/content/classroom-code.zip')
    with snapshot.open('wb') as handle:
        subprocess.run(['git','-C',str(REPO),'archive','--format=zip','HEAD'], stdout=handle, check=True, timeout=min(90,remaining()))
    LOCAL.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(snapshot) as archive:
        assert all(not Path(item.filename).is_absolute() and '..' not in Path(item.filename).parts for item in archive.infolist())
        archive.extractall(LOCAL)
    os.chdir(LOCAL)
    from scripts.colab_process import run_logged
    run_logged([sys.executable,'-m','pip','install','-q','-r','requirements-colab.txt'], OUTPUT/'setup.log',min(240,remaining(180)),revision=REVISION)
    print('Revision:',REVISION)
except BaseException:
    runtime.unassign()
    raise


In [ ]:
try:
    manifest = SOURCE/'source_manifest.json'
    annotations = REVIEW/'annotations.reviewed.json'
    run_logged([sys.executable,'-m','scripts.fight_classroom_refine','--phase','check','--manifest',str(manifest),'--annotations',str(annotations)], OUTPUT/'preflight.log',min(90,remaining(180)),revision=REVISION)
    base = ROOT/'artifacts/fight_large_v1'
    assert (base/'manifest.json').is_file(), 'Thiếu manifest split gốc; không tự chia lại.'
    run_logged([sys.executable,'-m','scripts.fight_public_data','--archives',str(ROOT/'datasets/fight'),'--data',str(DATA),'--output',str(base)],OUTPUT/'data.log',remaining(600),revision=REVISION)
    models = Path('/content/classroom-parent-models')
    models.mkdir(exist_ok=True)
    with zipfile.ZipFile(SOURCE/'rgb_aux/sentinel_fight_only_runtime.zip') as archive:
        for name in ('fight_model.ts','fight_runtime.json','yolov8n-pose.pt'):
            (models/name).write_bytes(archive.read('models/'+name))
    parent_local = Path('/content/classroom-parent-run')
    (parent_local/'rgb_control').mkdir(parents=True, exist_ok=True)
    shutil.copyfile(PARENT/'run.json',parent_local/'run.json')
    shutil.copyfile(PARENT/'rgb_control/best.pt',parent_local/'rgb_control/best.pt')
    CACHE.mkdir(exist_ok=True)
    saved_cache = OUTPUT/'window_cache.zip'
    if saved_cache.exists():
        with zipfile.ZipFile(saved_cache) as archive:
            assert all('/' not in item.filename and '\\' not in item.filename and item.filename.endswith('.npz') for item in archive.infolist())
            archive.extractall(CACHE)
    command = [sys.executable,'-m','scripts.fight_classroom_refine','--manifest',str(manifest),'--annotations',str(annotations),'--models',str(models),'--parent-run',str(parent_local),'--data',str(DATA),'--cache',str(CACHE),'--output',str(OUTPUT),'--device','cuda','--resume']
    provenance = OUTPUT/'provenance.json'
    if not provenance.exists():
        provenance.write_text(json.dumps({'revision':REVISION,'scope':'Public-only classroom repair; private phone recordings never uploaded; no auto-deployment'},indent=2))
    frozen = OUTPUT/'selection_frozen.json'
    if frozen.exists() and not json.loads(frozen.read_text())['selection']['improved']:
        raise RuntimeError('Run cũ không cải thiện development; không tự chạy thêm hoặc dùng test để chọn model.')
    phases = [('evaluate',12)] if frozen.exists() else [('prepare',10),('train',18),('evaluate',12)]
    for phase, cap in phases:
        minutes = min(cap, remaining(180)/60)
        assert minutes >= 1, 'Không đủ thời gian; giữ checkpoint để tiếp tục sau.'
        run_logged(command+['--phase',phase,'--minutes',str(minutes)],OUTPUT/(phase+'.log'),min(minutes*60+30,remaining(150)),revision=REVISION)
        status = json.loads((OUTPUT/(phase+'_status.json')).read_text())
        print('PHASE:',phase,status['status'],flush=True)
        if status['status'] != 'completed':
            print('Đã lưu tiến độ. Không tự retry hoặc thay model.',flush=True)
            break
        if phase == 'train' and not status['training']['improved']:
            print('Development không tiến triển: dừng, không dùng test để chọn lại model.',flush=True)
            break
    print('Artifacts:',OUTPUT,flush=True)
finally:
    try:
        if CACHE.exists():
            partial = OUTPUT/'window_cache.partial.zip'
            with zipfile.ZipFile(partial,'w',compression=zipfile.ZIP_STORED) as archive:
                for path in sorted(CACHE.glob('*.npz')):
                    if not path.name.endswith('.partial.npz'):
                        archive.write(path,path.name)
            partial.replace(OUTPUT/'window_cache.zip')
            print('Đã lưu cache public trên Drive.',flush=True)
    finally:
        print('Ngắt GPU; ứng dụng local chưa thay checkpoint.',flush=True)
        runtime.unassign()
